# LeetCode 662 — Maximum Width of Binary Tree bằng Python

Link đề: https://leetcode.com/problems/maximum-width-of-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def widthOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: trên LeetCode lớp `TreeNode` được cho sẵn và cây được truyền vào dưới dạng danh sách theo thứ tự tầng (level order) có `null`. Trong notebook này ta **tự dựng** `TreeNode`, hàm dựng cây từ danh sách kiểu LeetCode (`build_tree`) và hàm in cây ngược lại, để chạy mọi ví dụ ngay tại đây. Điểm mấu chốt của bài: "chiều rộng" **tính cả các vị trí trống (null) nằm giữa** hai đầu của một tầng, nên ta cần đánh số vị trí của nút như trong cây nhị phân đầy đủ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu định nghĩa **chiều rộng của một tầng** = khoảng cách giữa nút ngoài cùng bên trái và nút ngoài cùng bên phải của tầng đó, **tính cả các ô trống** ở giữa như trong cây nhị phân đầy đủ; đáp án là chiều rộng lớn nhất qua mọi tầng.
2. Nắm các công cụ Python cần dùng: lớp `TreeNode`, hàng đợi `collections.deque`, duyệt theo tầng bằng `for _ in range(len(q))`, phép dịch bit `<<` và `|`, `dict.setdefault`, và số nguyên **không giới hạn** của Python.
3. Đi từ **bung cây thành các tầng có `None`** → **BFS đánh số vị trí kiểu heap (`2i`, `2i+1`)** → **DFS ghi nhớ chỉ số đầu tiên của mỗi tầng** → **BFS với chỉ số chuẩn hóa để số không phình to**.
4. Hiểu mấu chốt: nút có chỉ số `i` thì con trái có `2i`, con phải có `2i + 1` (như mảng của cây đầy đủ); chiều rộng tầng = `chỉ số phải nhất - chỉ số trái nhất + 1`. **Không cần tạo ra các ô `None`.**
5. Tránh các lỗi kinh điển: chỉ đếm số nút của tầng, quên `+ 1`, lấy hiệu chỉ số trên toàn cây thay vì trong từng tầng, và ghi đè "chỉ số đầu tiên" của mỗi tầng trong DFS.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân, trả về **chiều rộng lớn nhất** của cây.

**Chiều rộng của một tầng** là độ dài giữa hai nút ngoài cùng (nút khác `null` trái nhất và phải nhất) của tầng đó; các nút `null` nằm giữa hai đầu mà **sẽ có mặt trong một cây nhị phân đầy đủ kéo dài tới tầng này** cũng được tính vào độ dài. Chiều rộng lớn nhất của cây là giá trị lớn nhất trong các tầng. Đảm bảo đáp án vừa trong số nguyên 32-bit có dấu.

**Ví dụ 1**

- Input: `root = [1,3,2,5,3,null,9]`
- Output: `4`
- Giải thích: tầng 3 có `5, 3, null, 9`, độ dài `4`.

**Ví dụ 2**

- Input: `root = [1,3,2,5,null,null,9,6,null,7]`
- Output: `7`
- Giải thích: tầng 4 là `6, null, null, null, null, null, 7`, độ dài `7`.

**Ví dụ 3**

- Input: `root = [1,3,2,5]`
- Output: `2`
- Giải thích: tầng 2 là `3, 2`, độ dài `2`.

**Ràng buộc:**

- Số nút trong cây thuộc đoạn `[1, 3000]`.
- `-100 <= Node.val <= 100`

Hệ quả: cây có thể sâu tới `3000` tầng (một dây chuyền), khi đó nếu đánh chỉ số kiểu heap thì chỉ số lên tới `2^3000` — số khổng lồ (ở Python không sao, ở C++/Java sẽ tràn).

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    # dựng cây từ danh sách theo thứ tự tầng, đúng quy ước của LeetCode
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    # ngược lại: cây -> danh sách theo thứ tự tầng, bỏ null thừa ở cuối
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, 3, 2, 5, 3, None, 9], 4),
    ([1, 3, 2, 5, None, None, 9, 6, None, 7], 7),
    ([1, 3, 2, 5], 2),
]
for vals, expected in examples:
    root = build_tree(vals)
    assert tree_to_list(root) == vals
    print(vals, "-> kỳ vọng", expected)

[1, 3, 2, 5, 3, None, 9] -> kỳ vọng 4
[1, 3, 2, 5, None, None, 9, 6, None, 7] -> kỳ vọng 7
[1, 3, 2, 5] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. `TreeNode` và dựng cây từ danh sách LeetCode

Mỗi nút có `val`, `left`, `right`. Danh sách đầu vào theo thứ tự tầng: phần tử `null` nghĩa là không có nút tại chỗ đó, và **nút `null` không có con** (không có hai phần tử `null` cho nó). Hàm `build_tree` ở trên dùng hàng đợi: lấy từng nút ra, gán hai phần tử kế tiếp của danh sách làm con trái/phải.

Hãy chạy thử để xem cây của ví dụ 1 trông thế nào.

In [2]:
def show(node, indent=0, tag="gốc"):
    if node is None:
        return
    print("  " * indent + f"{tag}: {node.val}")
    show(node.left, indent + 1, "L")
    show(node.right, indent + 1, "R")

show(build_tree([1, 3, 2, 5, 3, None, 9]))

gốc: 1
  L: 3
    L: 5
    R: 3
  R: 2
    R: 9


### 3.2. Duyệt theo tầng bằng `deque`

`collections.deque` là hàng đợi hai đầu: `popleft()` lấy đầu, `append()` thêm cuối, cả hai `O(1)`. Với `deque` ta truy cập đầu `q[0]` và cuối `q[-1]` cũng `O(1)` — rất tiện để lấy nút trái nhất và phải nhất của tầng.

Mẫu duyệt từng tầng: trước khi xử lý, ghi `n = len(q)` (số nút của tầng hiện tại), rồi lặp `n` lần: lấy ra một nút, đẩy con vào cuối. Sau `n` lần, `q` chỉ còn các nút của tầng kế tiếp.

In [3]:
root = build_tree([1, 3, 2, 5, 3, None, 9])
q = deque([root])
level = 1
while q:
    print("tầng", level, ":", [node.val for node in q], "| trái nhất =", q[0].val, ", phải nhất =", q[-1].val)
    for _ in range(len(q)):
        node = q.popleft()
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    level += 1

tầng 1 : [1] | trái nhất = 1 , phải nhất = 1
tầng 2 : [3, 2] | trái nhất = 3 , phải nhất = 2
tầng 3 : [5, 3, 9] | trái nhất = 5 , phải nhất = 9


### 3.3. Đánh số vị trí kiểu heap

Trong một **cây nhị phân đầy đủ** ta có thể đánh số các vị trí: gốc là `1`; nút ở vị trí `i` có con trái ở `2i` và con phải ở `2i + 1`; ngược lại cha của `i` là `i // 2`. Như vậy các vị trí của tầng `d` (gốc là tầng `0`) là các số từ `2^d` đến `2^(d+1) - 1`, **liên tiếp nhau**. Hiệu hai chỉ số cho biết số vị trí nằm giữa — kể cả vị trí trống.

Vì vậy chiều rộng của một tầng bằng `chỉ_số_phải_nhất - chỉ_số_trái_nhất + 1`. Ta **không cần** biết các vị trí trống ở giữa.

In [4]:
for i in [1, 2, 3, 4, 5, 6, 7]:
    print(f"vị trí {i}: con trái = {2 * i}, con phải = {2 * i + 1}, cha = {i // 2}")
print("tầng 3 (gốc là tầng 0) gồm các vị trí", list(range(2 ** 3, 2 ** 4)))

vị trí 1: con trái = 2, con phải = 3, cha = 0
vị trí 2: con trái = 4, con phải = 5, cha = 1
vị trí 3: con trái = 6, con phải = 7, cha = 1
vị trí 4: con trái = 8, con phải = 9, cha = 2
vị trí 5: con trái = 10, con phải = 11, cha = 2
vị trí 6: con trái = 12, con phải = 13, cha = 3
vị trí 7: con trái = 14, con phải = 15, cha = 3
tầng 3 (gốc là tầng 0) gồm các vị trí [8, 9, 10, 11, 12, 13, 14, 15]


### 3.4. Phép dịch bit `<<` và `|`

`i << 1` là dịch `i` sang trái 1 bit, tức nhân đôi: `i << 1 == 2 * i`. `i << 1 | 1` đặt thêm bit thấp nhất bằng `1`, tức `2 * i + 1` (vì bit thấp nhất của `i << 1` luôn là `0`). Về thứ tự ưu tiên, `<<` cao hơn `|`, nên `i << 1 | 1` được hiểu là `(i << 1) | 1`.

In [5]:
i = 5
print(i << 1, i << 1 | 1, (i << 1) | 1, 2 * i, 2 * i + 1)
print(bin(5), bin(5 << 1), bin(5 << 1 | 1))

10 11 11 10 11
0b101 0b1010 0b1011


### 3.5. Số nguyên lớn không tràn trong Python

Python có số nguyên **độ dài tùy ý**, nên `2 ** 3000` hoàn toàn bình thường (chỉ chậm hơn một chút khi cộng/trừ). Ở C++/Java số nguyên 32/64-bit sẽ tràn. Ta kiểm tra bằng `.bit_length()` (số bit cần để biểu diễn).

In [6]:
x = 1
for _ in range(2999):         # đi xuống 2999 lần "con trái": một dây chuyền 3000 nút
    x = x << 1
print("bit_length =", x.bit_length(), "| chữ số thập phân:", len(str(x)))
print((1 << 64).bit_length(), "-> vượt 64-bit, nhưng Python vẫn tính đúng")

bit_length = 3000 | chữ số thập phân: 903
65 -> vượt 64-bit, nhưng Python vẫn tính đúng


### 3.6. `dict.setdefault`

`d.setdefault(key, value)` trả `d[key]` nếu `key` đã có; nếu chưa có thì gán `d[key] = value` rồi trả `value`. Ta sẽ dùng nó để ghi nhớ **lần đầu tiên** gặp một tầng (chỉ số trái nhất) khi duyệt DFS.

In [7]:
first = {}
for depth, idx in [(0, 1), (1, 2), (1, 3), (2, 4), (2, 5)]:
    first.setdefault(depth, idx)      # chỉ lần đầu của mỗi depth được ghi
print(first)

{0: 1, 1: 2, 2: 4}


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hãy làm tay với **ví dụ 1** `[1,3,2,5,3,null,9]`. Vẽ cây "đầy đủ" và điền vị trí:

```
tầng 0:                 1(vt 1)
tầng 1:        3(vt 2)              2(vt 3)
tầng 2:   5(vt 4)  3(vt 5)    [trống vt 6]  9(vt 7)
```

- Tầng 0: một nút, vị trí `1`..`1`, chiều rộng `1`.
- Tầng 1: vị trí `2` và `3`, chiều rộng `3 - 2 + 1 = 2`.
- Tầng 2: nút trái nhất ở vị trí `4`, nút phải nhất ở vị trí `7`, chiều rộng `7 - 4 + 1 = 4` — gồm `5, 3, trống, 9`. Đúng như đề.

Đáp án là `max(1, 2, 4) = 4`.

Điều rút ra:

1. Chỉ cần biết **vị trí của nút trái nhất và phải nhất** mỗi tầng.
2. Vị trí của nút con tính từ vị trí của cha: `2i` hoặc `2i + 1`.
3. Nên duyệt theo tầng (BFS), mang theo vị trí; hoặc duyệt DFS (ưu tiên trái) và ghi nhớ vị trí đầu tiên gặp ở mỗi độ sâu.

In [8]:
# Tự động hóa việc tính tay: gán vị trí cho từng nút và in theo tầng
def positions_by_level(root):
    levels = {}
    def dfs(node, depth, i):
        if node is None:
            return
        levels.setdefault(depth, []).append((node.val, i))
        dfs(node.left, depth + 1, 2 * i)
        dfs(node.right, depth + 1, 2 * i + 1)
    dfs(root, 0, 1)
    return levels

for vals, _ in examples:
    print(vals)
    for d, items in positions_by_level(build_tree(vals)).items():
        pos = sorted(p for _, p in items)
        print(f"   tầng {d}: (val, vị trí) = {sorted(items, key=lambda t: t[1])} -> rộng {pos[-1] - pos[0] + 1}")

[1, 3, 2, 5, 3, None, 9]
   tầng 0: (val, vị trí) = [(1, 1)] -> rộng 1
   tầng 1: (val, vị trí) = [(3, 2), (2, 3)] -> rộng 2
   tầng 2: (val, vị trí) = [(5, 4), (3, 5), (9, 7)] -> rộng 4
[1, 3, 2, 5, None, None, 9, 6, None, 7]
   tầng 0: (val, vị trí) = [(1, 1)] -> rộng 1
   tầng 1: (val, vị trí) = [(3, 2), (2, 3)] -> rộng 2
   tầng 2: (val, vị trí) = [(5, 4), (9, 7)] -> rộng 4
   tầng 3: (val, vị trí) = [(6, 8), (7, 14)] -> rộng 7
[1, 3, 2, 5]
   tầng 0: (val, vị trí) = [(1, 1)] -> rộng 1
   tầng 1: (val, vị trí) = [(3, 2), (2, 3)] -> rộng 2
   tầng 2: (val, vị trí) = [(5, 4)] -> rộng 1


## 5. Cách 1 — Brute Force: bung từng tầng ra danh sách có `None`

### Ý tưởng

Làm đúng theo lời đề: với mỗi tầng ta giữ **danh sách đầy đủ các vị trí**, gồm cả `None` ở những chỗ trống. Từ danh sách tầng hiện tại, tầng kế tiếp được tạo bằng cách thay mỗi phần tử bằng hai con của nó (`None` cho ra hai `None`). Chiều rộng tầng là độ dài đoạn từ phần tử khác `None` đầu tiên tới phần tử khác `None` cuối cùng.

Để danh sách đỡ dài vô ích, mỗi tầng ta cắt bỏ các `None` ở hai đầu (không làm đổi khoảng cách tương đối).

### Code

In [9]:
def width_expand(root):
    level = [root]
    best = 0
    while level:
        # cắt None hai đầu
        lo = 0
        while lo < len(level) and level[lo] is None:
            lo += 1
        hi = len(level) - 1
        while hi >= 0 and level[hi] is None:
            hi -= 1
        if lo > hi:                       # toàn None: hết cây
            break
        level = level[lo:hi + 1]
        best = max(best, len(level))
        nxt = []
        for node in level:
            if node is None:
                nxt += [None, None]
            else:
                nxt += [node.left, node.right]
        level = nxt
    return best

for vals, expected in examples:
    print(vals, "->", width_expand(build_tree(vals)), "(kỳ vọng", expected, ")")

[1, 3, 2, 5, 3, None, 9] -> 4 (kỳ vọng 4 )
[1, 3, 2, 5, None, None, 9, 6, None, 7] -> 7 (kỳ vọng 7 )
[1, 3, 2, 5] -> 2 (kỳ vọng 2 )


### Độ phức tạp và nút thắt

- Time/Space: bằng tổng độ dài các danh sách tầng, tức tổng các chiều rộng. Chiều rộng có thể lên tới `2^(h-1)` với `h` là chiều cao. Với `n` nút thì tệ nhất cỡ `O(2^(n/2))`: **mũ** theo số nút!

**Nút thắt:** ta tốn công lưu từng ô trống, trong khi chỉ cần biết **hai đầu**. Thử với cây "hai dây chuyền": một dây chuyền đi xuống bên trái và một đi xuống bên phải; chỉ có `2d - 1` nút nhưng chiều rộng tầng cuối là `2^(d-1)`.

In [10]:
def two_chains(d):
    # gốc có một dây chuyền trái dài d-1 nút và một dây chuyền phải dài d-1 nút
    root = TreeNode(1)
    a = b = root
    for _ in range(d - 1):
        a.left = TreeNode(1); a = a.left
        b.right = TreeNode(1); b = b.right
    return root

for d in [6, 10, 14, 18, 20]:
    r = two_chains(d)
    t0 = time.perf_counter()
    w = width_expand(r)
    print(f"d={d}: {2 * d - 1} nút, chiều rộng {w}, thời gian {time.perf_counter() - t0:.4f}s")

d=6: 11 nút, chiều rộng 32, thời gian 0.0000s
d=10: 19 nút, chiều rộng 512, thời gian 0.0003s
d=14: 27 nút, chiều rộng 8192, thời gian 0.0039s
d=18: 35 nút, chiều rộng 131072, thời gian 0.0697s


d=20: 39 nút, chiều rộng 524288, thời gian 0.2726s


Số nút tăng tuyến tính nhưng thời gian tăng gấp đôi mỗi khi `d` tăng thêm `1`. Với `d = 3000` thì hoàn toàn không thể (và đáp án thật sự sẽ vượt 32-bit — đề đảm bảo không xảy ra, nhưng cây sâu mà hẹp vẫn tồn tại).

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là **không lưu các ô trống**, chỉ lưu **chỉ số vị trí** (theo mục 3.3) của các nút thật sự có mặt. Khi đó:

> Mỗi nút mang theo một số nguyên `i` (vị trí trong cây đầy đủ). Con trái có `2i`, con phải có `2i + 1`. Trong một tầng, chiều rộng = `i_cuối - i_đầu + 1`.

Với BFS: tại mỗi tầng, hàng đợi chứa các cặp `(nút, i)` theo thứ tự từ trái sang phải, nên `q[0]` là nút trái nhất và `q[-1]` là phải nhất. Ta cập nhật `ans = max(ans, q[-1][1] - q[0][1] + 1)` **trước** khi xử lý tầng.

Ta vẫn chỉ duyệt mỗi nút một lần, thời gian `O(n)`, bộ nhớ `O(độ rộng thật sự của hàng đợi) = O(n)`.

In [11]:
root = build_tree([1, 3, 2, 5, 3, None, 9])
q = deque([(root, 1)])
ans = 0
while q:
    width = q[-1][1] - q[0][1] + 1
    ans = max(ans, width)
    print("hàng đợi (val, i):", [(n.val, i) for n, i in q], "-> rộng", width)
    for _ in range(len(q)):
        node, i = q.popleft()
        if node.left:
            q.append((node.left, i << 1))
        if node.right:
            q.append((node.right, i << 1 | 1))
print("đáp án:", ans)

hàng đợi (val, i): [(1, 1)] -> rộng 1
hàng đợi (val, i): [(3, 2), (2, 3)] -> rộng 2
hàng đợi (val, i): [(5, 4), (3, 5), (9, 7)] -> rộng 4
đáp án: 4


## 7. Các cách tiếp theo

Ta có 4 cách khác nhau thực sự:

| Cách | Ý tưởng |
|---|---|
| 1 | Bung từng tầng có `None` (mục 5) |
| 2 | **BFS** mang theo chỉ số vị trí `(nút, i)` (cách nộp) |
| 3 | **DFS** (ưu tiên trái) ghi nhớ chỉ số đầu tiên của mỗi độ sâu |
| 4 | BFS với chỉ số **chuẩn hóa** (trừ chỉ số trái nhất của tầng) để số luôn nhỏ |

### Cách 2 — BFS với chỉ số

**Giải thích từng đoạn:**

- `q = deque([(root, 1)])`: hàng đợi các cặp `(nút, chỉ số)`, gốc có chỉ số `1`.
- Đầu mỗi vòng `while`: `q` đang chứa **đúng một tầng**; đo chiều rộng bằng `q[-1][1] - q[0][1] + 1`.
- `for _ in range(len(q))`: xử lý đúng các nút của tầng hiện tại; đẩy con trái với `i << 1`, con phải với `i << 1 | 1`.
- Nút `None` không bao giờ được đẩy vào hàng đợi.

### Code demo

In [12]:
def width_bfs(root):
    ans = 0
    q = deque([(root, 1)])
    while q:
        ans = max(ans, q[-1][1] - q[0][1] + 1)
        for _ in range(len(q)):
            node, i = q.popleft()
            if node.left:
                q.append((node.left, i << 1))
            if node.right:
                q.append((node.right, i << 1 | 1))
    return ans

for vals, expected in examples:
    print(vals, "->", width_bfs(build_tree(vals)), "(kỳ vọng", expected, ")")

[1, 3, 2, 5, 3, None, 9] -> 4 (kỳ vọng 4 )
[1, 3, 2, 5, None, None, 9, 6, None, 7] -> 7 (kỳ vọng 7 )
[1, 3, 2, 5] -> 2 (kỳ vọng 2 )


### Code nộp LeetCode (Cách 2)

```python
from collections import deque

class Solution:
    def widthOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        ans = 0
        q = deque([(root, 1)])
        while q:
            ans = max(ans, q[-1][1] - q[0][1] + 1)
            for _ in range(len(q)):
                node, i = q.popleft()
                if node.left:
                    q.append((node.left, i << 1))
                if node.right:
                    q.append((node.right, i << 1 | 1))
        return ans
```

### Độ phức tạp

- Time: `O(n)`, mỗi nút vào/ra hàng đợi một lần (các phép trên số lớn tốn thêm `O(độ dài bit)` nhưng với Python và `n <= 3000` vẫn rất nhanh).
- Space: `O(n)` cho hàng đợi (tệ nhất một tầng chứa gần `n/2` nút).

### Cách 3 — DFS ghi nhớ chỉ số đầu tiên của mỗi độ sâu

**Ý tưởng khác biệt:** không cần duyệt theo tầng. Duyệt DFS **ưu tiên trái** (tiền thứ tự: nút, trái, phải). Khi đó **lần đầu tiên** ta gặp một độ sâu `d`, nút đó chính là nút **trái nhất** của tầng `d`. Lưu `first[d] = i` ở lần đầu; với mỗi nút gặp sau đó tại cùng độ sâu, chiều rộng ứng viên là `i - first[d] + 1` (nút đang xét có thể là nút phải nhất hiện tại).

### Code demo

In [13]:
def width_dfs(root):
    first = {}
    best = 0

    def dfs(node, depth, i):
        nonlocal best
        if node is None:
            return
        first.setdefault(depth, i)                    # lần đầu gặp độ sâu này = nút trái nhất
        best = max(best, i - first[depth] + 1)
        dfs(node.left, depth + 1, 2 * i)
        dfs(node.right, depth + 1, 2 * i + 1)

    dfs(root, 0, 1)
    return best

for vals, expected in examples:
    print(vals, "->", width_dfs(build_tree(vals)), "(kỳ vọng", expected, ")")

[1, 3, 2, 5, 3, None, 9] -> 4 (kỳ vọng 4 )
[1, 3, 2, 5, None, None, 9, 6, None, 7] -> 7 (kỳ vọng 7 )
[1, 3, 2, 5] -> 2 (kỳ vọng 2 )


### Code nộp LeetCode (Cách 3)

```python
class Solution:
    def widthOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        first = {}
        best = 0

        def dfs(node, depth, i):
            nonlocal best
            if node is None:
                return
            first.setdefault(depth, i)
            best = max(best, i - first[depth] + 1)
            dfs(node.left, depth + 1, 2 * i)
            dfs(node.right, depth + 1, 2 * i + 1)

        dfs(root, 0, 1)
        return best
```

Lưu ý: cây có thể sâu tới `3000` tầng, vượt giới hạn đệ quy mặc định (`1000`) của Python; nếu gặp `RecursionError` hãy nâng bằng `sys.setrecursionlimit(10000)` hoặc dùng Cách 2 (BFS, không đệ quy).

### Độ phức tạp

- Time: `O(n)`.
- Space: `O(h)` cho ngăn xếp đệ quy và `O(h)` cho `first`, với `h` là chiều cao (tệ nhất `O(n)`), thường nhỏ hơn hàng đợi của BFS khi cây rộng và thấp.

### Cách 4 — BFS với chỉ số chuẩn hóa

**Vấn đề:** ở C++/Java, chỉ số kiểu heap của cây sâu sẽ **tràn** số nguyên. Đề đảm bảo **chiều rộng** (đáp án) vừa 32-bit, nhưng chính chỉ số thì không. Cách sửa: mỗi tầng, trừ đi chỉ số của nút trái nhất (`base`), để chỉ số tương đối của tầng luôn trong `[0, chiều_rộng - 1]`. Con có chỉ số tương đối `2 * (i - base)` hoặc `2 * (i - base) + 1`. Chiều rộng không đổi vì ta trừ cùng một hằng số cho cả tầng.

### Code demo

In [14]:
def width_normalized(root):
    ans = 0
    biggest = 0
    q = deque([(root, 0)])
    while q:
        base = q[0][1]
        ans = max(ans, q[-1][1] - base + 1)
        for _ in range(len(q)):
            node, i = q.popleft()
            i -= base                                   # chuẩn hóa: chỉ số tương đối trong tầng
            if node.left:
                q.append((node.left, 2 * i))
            if node.right:
                q.append((node.right, 2 * i + 1))
            biggest = max(biggest, 2 * i + 1)
    return ans, biggest

for vals, expected in examples:
    print(vals, "->", width_normalized(build_tree(vals))[0], "(kỳ vọng", expected, ")")

# Dây chuyền trái 3000 nút: chỉ số heap thô lên tới 2^2999, chỉ số chuẩn hóa vẫn nhỏ
chain = TreeNode(1)
cur = chain
for _ in range(2999):
    cur.left = TreeNode(1); cur = cur.left
w, big = width_normalized(chain)
print("dây chuyền 3000 nút: rộng", w, "| chỉ số lớn nhất khi chuẩn hóa:", big)
print("còn chỉ số heap thô cần", (1 << 2999).bit_length(), "bit")

[1, 3, 2, 5, 3, None, 9] -> 4 (kỳ vọng 4 )
[1, 3, 2, 5, None, None, 9, 6, None, 7] -> 7 (kỳ vọng 7 )
[1, 3, 2, 5] -> 2 (kỳ vọng 2 )
dây chuyền 3000 nút: rộng 1 | chỉ số lớn nhất khi chuẩn hóa: 1
còn chỉ số heap thô cần 3000 bit


### Code nộp LeetCode (Cách 4)

```python
from collections import deque

class Solution:
    def widthOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        ans = 0
        q = deque([(root, 0)])
        while q:
            base = q[0][1]
            ans = max(ans, q[-1][1] - base + 1)
            for _ in range(len(q)):
                node, i = q.popleft()
                i -= base
                if node.left:
                    q.append((node.left, 2 * i))
                if node.right:
                    q.append((node.right, 2 * i + 1))
        return ans
```

### Độ phức tạp

- Time: `O(n)`.
- Space: `O(n)`; mọi chỉ số `< 2 * chiều_rộng`, an toàn với kiểu 64-bit (vì chiều rộng vừa 32-bit).

## 8. TRACE từng bước

Hàm `trace_width(root)` in từng tầng: danh sách `(val, i)` trong hàng đợi, chỉ số trái nhất/phải nhất, chiều rộng và giá trị `ans` hiện tại.

Bảng trạng thái cho **ví dụ 1** `[1,3,2,5,3,null,9]`:

| Tầng | Hàng đợi `(val, i)` | `i` đầu | `i` cuối | Rộng | `ans` |
|---|---|---|---|---|---|
| 0 | `(1,1)` | 1 | 1 | 1 | 1 |
| 1 | `(3,2) (2,3)` | 2 | 3 | 2 | 2 |
| 2 | `(5,4) (3,5) (9,7)` | 4 | 7 | 4 | 4 |

In [15]:
def trace_width(root):
    ans = 0
    q = deque([(root, 1)])
    level = 0
    while q:
        left_i, right_i = q[0][1], q[-1][1]
        width = right_i - left_i + 1
        ans = max(ans, width)
        print(f"tầng {level}: hàng đợi = {[(n.val, i) for n, i in q]}")
        print(f"          i đầu = {left_i}, i cuối = {right_i} -> rộng = {right_i} - {left_i} + 1 = {width} | ans = {ans}")
        for _ in range(len(q)):
            node, i = q.popleft()
            if node.left:
                q.append((node.left, i << 1))
            if node.right:
                q.append((node.right, i << 1 | 1))
        level += 1
    return ans

print("=>", trace_width(build_tree([1, 3, 2, 5, 3, None, 9])))

tầng 0: hàng đợi = [(1, 1)]
          i đầu = 1, i cuối = 1 -> rộng = 1 - 1 + 1 = 1 | ans = 1
tầng 1: hàng đợi = [(3, 2), (2, 3)]
          i đầu = 2, i cuối = 3 -> rộng = 3 - 2 + 1 = 2 | ans = 2
tầng 2: hàng đợi = [(5, 4), (3, 5), (9, 7)]
          i đầu = 4, i cuối = 7 -> rộng = 7 - 4 + 1 = 4 | ans = 4
=> 4


In [16]:
print("=>", trace_width(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])))

tầng 0: hàng đợi = [(1, 1)]
          i đầu = 1, i cuối = 1 -> rộng = 1 - 1 + 1 = 1 | ans = 1
tầng 1: hàng đợi = [(3, 2), (2, 3)]
          i đầu = 2, i cuối = 3 -> rộng = 3 - 2 + 1 = 2 | ans = 2
tầng 2: hàng đợi = [(5, 4), (9, 7)]
          i đầu = 4, i cuối = 7 -> rộng = 7 - 4 + 1 = 4 | ans = 4
tầng 3: hàng đợi = [(6, 8), (7, 14)]
          i đầu = 8, i cuối = 14 -> rộng = 14 - 8 + 1 = 7 | ans = 7
=> 7


Ở ví dụ 2: tầng 3 chứa nút `6` (con trái của `5` ở vị trí `4`, nên có vị trí `8`) và nút `7` (con trái của `9` ở vị trí `7`, nên có vị trí `14`). Chiều rộng là `14 - 8 + 1 = 7`, khớp đáp án. Chú ý các vị trí `9..13` ở giữa **không có nút nào** nhưng vẫn được tính vào chiều rộng — đó là ý nghĩa của "đếm cả null ở giữa".

## 9. Code nộp LeetCode

Ta nộp Cách 2 (BFS với chỉ số): ngắn gọn, không đệ quy, `O(n)`.

In [17]:
class Solution:
    def widthOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        ans = 0
        q = deque([(root, 1)])
        while q:
            ans = max(ans, q[-1][1] - q[0][1] + 1)
            for _ in range(len(q)):
                node, i = q.popleft()
                if node.left:
                    q.append((node.left, i << 1))
                if node.right:
                    q.append((node.right, i << 1 | 1))
        return ans


for vals, expected in examples:
    got = Solution().widthOfBinaryTree(build_tree(vals))
    print(vals, "->", got, "| kỳ vọng:", expected, "| OK" if got == expected else "| SAI")

[1, 3, 2, 5, 3, None, 9] -> 4 | kỳ vọng: 4 | OK
[1, 3, 2, 5, None, None, 9, 6, None, 7] -> 7 | kỳ vọng: 7 | OK
[1, 3, 2, 5] -> 2 | kỳ vọng: 2 | OK


## 10. Vì sao đúng

**(a) Chỉ số là đơn ánh trong từng tầng.** Trong cây nhị phân đầy đủ mỗi vị trí có một chỉ số duy nhất (gốc `1`, con `2i`, `2i + 1`). Với cây của ta, mỗi nút thật được gắn đúng chỉ số vị trí mà nó chiếm khi "nhúng" cây vào cây đầy đủ. Hai nút khác nhau cùng tầng luôn có chỉ số khác nhau.

**(b) Thứ tự chỉ số trong tầng trùng thứ tự trái-phải.** Các chỉ số của tầng `d` là `2^d, ..., 2^(d+1) - 1`, tăng dần từ trái sang phải. BFS theo thứ tự "con trái trước con phải" giữ đúng thứ tự trái-phải nên `q[0]` có chỉ số nhỏ nhất, `q[-1]` có chỉ số lớn nhất.

**(c) Chiều rộng.** Số vị trí (kể cả trống) nằm giữa hai nút ngoài cùng, tính cả hai nút, là `i_cuối - i_đầu + 1` vì các vị trí liên tiếp có chỉ số liên tiếp.

**(d) Cách 4 giữ nguyên chiều rộng.** Ở mỗi tầng, mọi nút bị trừ **cùng một hằng số** `base`, nên hiệu số chỉ số giữa hai nút cùng tầng không đổi. Chỉ số của con được tính từ chỉ số tương đối của cha là `2j` hoặc `2j + 1`, tương ứng với chỉ số heap `2(j + base)` hoặc `2(j + base) + 1` trừ đi `2 * base`: tức là các con của tầng sau cũng bị lệch **cùng một hằng số** `2 * base`. Vậy chỉ số tương đối ở tầng sau vẫn là chỉ số heap thật trừ một hằng số chung của tầng đó, nên hiệu số (và chiều rộng) không đổi.

**(e) Cách 3 (DFS):** duyệt tiền thứ tự trái trước phải gặp các nút của một tầng theo thứ tự từ trái sang phải, nên nút đầu tiên của độ sâu `d` là nút trái nhất; mọi nút sau có chỉ số lớn hơn, nên `max` của `i - first[d] + 1` chính là `i_phải_nhất - i_trái_nhất + 1`.

Kiểm chứng bằng thí nghiệm: một "oracle" thứ hai tính vị trí bằng **chuỗi đường đi** (không dùng phép nhân đôi) rồi so với BFS trên nhiều cây ngẫu nhiên.

In [18]:
def random_tree(n, rng, max_depth=10, deep_bias=0.5):
    # cây ngẫu nhiên n nút, độ sâu <= max_depth; deep_bias lớn thì cây lệch/sâu hơn
    root = TreeNode(rng.randint(-100, 100))
    slots = [(root, "left", 1), (root, "right", 1)]
    count = 1
    while count < n and slots:
        if rng.random() < deep_bias:
            k = len(slots) - 1 - rng.randrange(min(2, len(slots)))
        else:
            k = rng.randrange(len(slots))
        parent, side, depth = slots.pop(k)
        child = TreeNode(rng.randint(-100, 100))
        setattr(parent, side, child)
        count += 1
        if depth + 1 <= max_depth:
            slots.append((child, "left", depth + 1))
            slots.append((child, "right", depth + 1))
    return root


def width_path_oracle(root):
    # vị trí của nút = số nhị phân "1" + dãy bước (0 = trái, 1 = phải): tính bằng chuỗi, không nhân đôi
    best = 0
    levels = {}
    stack = [(root, "1")]
    while stack:
        node, path = stack.pop()
        levels.setdefault(len(path), []).append(int(path, 2))
        if node.left:
            stack.append((node.left, path + "0"))
        if node.right:
            stack.append((node.right, path + "1"))
    for pos in levels.values():
        best = max(best, max(pos) - min(pos) + 1)
    return best


rng = random.Random(662)
cnt = 0
for _ in range(500):
    t = random_tree(rng.randint(1, 40), rng, max_depth=rng.choice([4, 8, 12]), deep_bias=rng.choice([0.1, 0.5, 0.9]))
    assert Solution().widthOfBinaryTree(t) == width_path_oracle(t) == width_expand(t)
    cnt += 1
print("oracle theo đường đi, Cách 1 và BFS khớp trên", cnt, "cây ngẫu nhiên")

oracle theo đường đi, Cách 1 và BFS khớp trên 500 cây ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 Bung có `None` | `O(tổng chiều rộng các tầng)` — có thể mũ theo `n` | cùng bậc |
| 2 BFS với chỉ số | `O(n)` | `O(n)` |
| 3 DFS + `first[depth]` | `O(n)` | `O(h)` |
| 4 BFS chuẩn hóa | `O(n)` | `O(n)` |

Ta đo thời gian thực của Cách 2, 3, 4 trên cây lớn `3000` nút (một dây chuyền, một cây ngẫu nhiên, và một cây gần đầy đủ).

In [19]:
sys.setrecursionlimit(20000)       # cho Cách 3 (DFS) chạy được trên dây chuyền sâu 3000


def full_tree(n):
    # cây gần đầy đủ n nút: nút i có con 2i+1, 2i+2 (dựng bằng danh sách)
    return build_tree(list(range(1, n + 1)))

chain = TreeNode(1); cur = chain
for _ in range(2999):
    cur.left = TreeNode(1); cur = cur.left

rng = random.Random(3)
cases = {"dây chuyền 3000": chain, "ngẫu nhiên 3000": random_tree(3000, rng, max_depth=3000, deep_bias=0.3),
         "gần đầy đủ 3000": full_tree(3000)}
for name, t in cases.items():
    res = []
    for fn in (width_bfs, width_dfs, lambda r: width_normalized(r)[0]):
        t0 = time.perf_counter()
        res.append((fn(t), round(time.perf_counter() - t0, 4)))
    assert res[0][0] == res[1][0] == res[2][0]
    print(f"{name}: rộng {res[0][0]} | BFS {res[0][1]}s, DFS {res[1][1]}s, chuẩn hóa {res[2][1]}s")

dây chuyền 3000: rộng 1 | BFS 0.002s, DFS 0.0032s, chuẩn hóa 0.0016s
ngẫu nhiên 3000: rộng 359080502 | BFS 0.0007s, DFS 0.0012s, chuẩn hóa 0.001s
gần đầy đủ 3000: rộng 1024 | BFS 0.0008s, DFS 0.0012s, chuẩn hóa 0.0008s


## 12. Bộ test

Các nhóm test: ví dụ của đề, edge case (một nút, dây chuyền, hai dây chuyền, cây đầy đủ, cây thiếu nút giữa), phản ví dụ (các trường hợp đếm sai nếu quên null), và đối chiếu **cả 4 cách** + oracle độc lập trên cây ngẫu nhiên có seed, trong ràng buộc `1..3000` nút.

In [20]:
# 1) Ví dụ của đề
for vals, expected in examples:
    assert Solution().widthOfBinaryTree(build_tree(vals)) == expected

# 2) Edge case
assert Solution().widthOfBinaryTree(TreeNode(7)) == 1                         # một nút
assert Solution().widthOfBinaryTree(build_tree([1, 2])) == 1                  # chỉ con trái
assert Solution().widthOfBinaryTree(build_tree([1, None, 2])) == 1            # chỉ con phải
assert Solution().widthOfBinaryTree(build_tree([1, 2, 3])) == 2
assert Solution().widthOfBinaryTree(build_tree([1, 2, 3, 4, None, None, 5, 6, None, None, 7])) == 8
for d in range(1, 12):
    full = build_tree([1] * (2 ** d - 1))                                       # cây đầy đủ d tầng
    assert Solution().widthOfBinaryTree(full) == 2 ** (d - 1), d
for d in range(2, 16):
    assert Solution().widthOfBinaryTree(two_chains(d)) == 2 ** (d - 1), d       # hai dây chuyền
print("ví dụ + edge nhỏ OK")

# 3) Cây rất sâu và hẹp: chỉ BFS/ chuẩn hóa (không đệ quy) là an toàn tuyệt đối
chain_l = TreeNode(1); cur = chain_l
for k in range(2999):
    if k % 2 == 0:
        cur.left = TreeNode(1); cur = cur.left
    else:
        cur.right = TreeNode(1); cur = cur.right      # zigzag
assert Solution().widthOfBinaryTree(chain_l) == 1
assert width_normalized(chain_l)[0] == 1
print("dây chuyền zigzag 3000 nút -> rộng 1 OK")

ví dụ + edge nhỏ OK
dây chuyền zigzag 3000 nút -> rộng 1 OK


In [21]:
# 4) Đối chiếu nhiều cách trên cây ngẫu nhiên (độ sâu hạn chế để Cách 1 chạy được)
rng = random.Random(662)
total = 0
for _ in range(1500):
    n = rng.randint(1, 60)
    t = random_tree(n, rng, max_depth=rng.choice([3, 6, 10, 13]), deep_bias=rng.choice([0.0, 0.3, 0.7, 0.95]))
    a = Solution().widthOfBinaryTree(t)
    assert a == width_bfs(t) == width_dfs(t) == width_normalized(t)[0] == width_expand(t) == width_path_oracle(t)
    total += 1
print("đối chiếu 6 cách trên", total, "cây ngẫu nhiên: OK")

# 5) Cây lớn hơn (không dùng Cách 1): BFS, DFS, chuẩn hóa, oracle đường đi
rng = random.Random(6620)
for _ in range(60):
    t = random_tree(rng.randint(500, 3000), rng, max_depth=rng.choice([20, 60, 200]), deep_bias=rng.choice([0.2, 0.6]))
    a = Solution().widthOfBinaryTree(t)
    assert a == width_dfs(t) == width_normalized(t)[0] == width_path_oracle(t)
print("cây lớn OK")

# 6) Đầu vào không bị sửa
t = build_tree([1, 3, 2, 5, 3, None, 9])
before = tree_to_list(t)
Solution().widthOfBinaryTree(t)
assert tree_to_list(t) == before
print("cây đầu vào không bị sửa")

đối chiếu 6 cách trên 1500 cây ngẫu nhiên: OK


cây lớn OK
cây đầu vào không bị sửa


## 13. Các lỗi người mới hay gặp

Ta cài vài hàm sai cố ý rồi tìm cây ngẫu nhiên khiến chúng sai bằng `find_mismatch`.

In [22]:
def find_mismatch(fn, trials=2000, seed=13):
    rng = random.Random(seed)
    best = None
    for _ in range(trials):
        t = random_tree(rng.randint(2, 12), rng, max_depth=6, deep_bias=rng.choice([0.2, 0.6]))
        if fn(t) != width_bfs(t):
            lst = tree_to_list(t)
            if best is None or len(lst) < len(best):
                best = lst
    return best

### Lỗi 1: chỉ đếm số nút của tầng, quên các ô trống ở giữa

Viết `width = len(q)` (số nút thật). Cách này trả `3` ở tầng 2 của ví dụ 1 thay vì `4`.

In [23]:
def wrong_count_nodes(root):
    ans = 0
    q = deque([root])
    while q:
        ans = max(ans, len(q))                  # SAI: bỏ qua các vị trí null nằm giữa
        for _ in range(len(q)):
            node = q.popleft()
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    return ans

print("sai :", wrong_count_nodes(build_tree([1, 3, 2, 5, 3, None, 9])), "| đúng:", Solution().widthOfBinaryTree(build_tree([1, 3, 2, 5, 3, None, 9])))
print("cây nhỏ nhất làm sai:", find_mismatch(wrong_count_nodes))

sai : 3 | đúng: 4
cây nhỏ nhất làm sai: [47, 9, -89, 56, None, -50]


**Vì sao sai:** đề yêu cầu đếm cả `null` nằm giữa; `len(q)` chỉ đếm nút thật. **Code đúng:** mang theo chỉ số vị trí và lấy hiệu `q[-1][1] - q[0][1] + 1`.

### Lỗi 2: quên `+ 1` (lệch một)

Hiệu hai chỉ số cho **số bước** giữa hai nút, còn chiều rộng đếm **số vị trí** (cả hai đầu).

In [24]:
def wrong_off_by_one(root):
    ans = 0
    q = deque([(root, 1)])
    while q:
        ans = max(ans, q[-1][1] - q[0][1])      # SAI: thiếu + 1
        for _ in range(len(q)):
            node, i = q.popleft()
            if node.left:
                q.append((node.left, i << 1))
            if node.right:
                q.append((node.right, i << 1 | 1))
    return ans

print("sai :", wrong_off_by_one(build_tree([1, 3, 2, 5, 3, None, 9])), "| đúng: 4")
print("một nút:", wrong_off_by_one(TreeNode(1)), "(đúng phải là 1, sai trả 0)")

sai : 3 | đúng: 4
một nút: 0 (đúng phải là 1, sai trả 0)


**Vì sao sai:** tầng chỉ có một nút có `i_cuối - i_đầu = 0` nhưng chiều rộng là `1`. **Code đúng:** `q[-1][1] - q[0][1] + 1`.

### Lỗi 3: lấy hiệu chỉ số trên **toàn cây** thay vì trong từng tầng

Chiều rộng được định nghĩa **theo từng tầng**. Nếu gom mọi chỉ số của cả cây rồi lấy `max - min + 1` thì ta đo khoảng cách giữa nút ở tầng sâu bên phải và nút gốc, hoàn toàn khác đề.

In [25]:
def wrong_global(root):
    lo, hi = None, 0
    q = deque([(root, 1)])
    while q:
        for _ in range(len(q)):
            node, i = q.popleft()
            lo = i if lo is None else min(lo, i)      # SAI: gộp mọi tầng
            hi = max(hi, i)
            if node.left:
                q.append((node.left, 2 * i))
            if node.right:
                q.append((node.right, 2 * i + 1))
    return hi - lo + 1

print("sai :", wrong_global(build_tree([1, 3, 2, 5, 3, None, 9])), "| đúng: 4")
print("cây nhỏ nhất làm sai:", find_mismatch(wrong_global))

sai : 7 | đúng: 4
cây nhỏ nhất làm sai: [93, 40]


**Vì sao sai:** chỉ số nhỏ nhất toàn cây luôn là `1` (gốc) nên kết quả chỉ là `chỉ số lớn nhất`, không liên quan chiều rộng của một tầng. **Code đúng:** tính `q[-1][1] - q[0][1] + 1` **cho từng tầng** rồi lấy `max`.

Ghi chú: nhiều bạn thắc mắc "gốc đánh số `0` với con `2i`, `2i + 1` có sai không?" Thực ra **không sai**: các chỉ số của tầng `d` khi đó là `0..2^d - 1`, vẫn liên tiếp và đúng thứ tự trái-phải, mà ta chỉ so sánh chỉ số **trong cùng một tầng**. Chuẩn heap (gốc `1`) chỉ là quy ước quen thuộc; Cách 4 ở trên chính là dùng chỉ số tương đối kiểu `0`.

### Lỗi 4: ghi đè "chỉ số đầu tiên" của mỗi tầng trong DFS

Dùng `first[depth] = i` (gán thẳng) thay vì chỉ ghi ở lần đầu (`setdefault`). Khi đó `first[depth]` luôn là chỉ số của nút **vừa duyệt**, không phải nút trái nhất.

In [26]:
def wrong_dfs_overwrite(root):
    first = {}
    best = 0
    def dfs(node, depth, i):
        nonlocal best
        if node is None:
            return
        first[depth] = i                              # SAI: ghi đè mỗi lần
        best = max(best, i - first[depth] + 1)       # luôn bằng 1
        dfs(node.left, depth + 1, 2 * i)
        dfs(node.right, depth + 1, 2 * i + 1)
    dfs(root, 0, 1)
    return best

print("sai :", wrong_dfs_overwrite(build_tree([1, 3, 2, 5, 3, None, 9])), "| đúng: 4")
print("cây nhỏ nhất làm sai:", find_mismatch(wrong_dfs_overwrite))

sai : 1 | đúng: 4
cây nhỏ nhất làm sai: [26, 43, -73]


**Vì sao sai:** `i - first[depth] + 1` luôn bằng `1` vì `first[depth] == i` ngay sau khi ghi đè. **Code đúng:** `first.setdefault(depth, i)` để chỉ nhớ **lần đầu tiên** (nút trái nhất khi duyệt trái trước).

## 14. Mẹo & nhận dạng pattern

- Thấy "chiều rộng tính cả null ở giữa", "vị trí trong cây đầy đủ", "kiểm tra cây đầy đủ/hoàn chỉnh" → **đánh chỉ số heap** (`2i`, `2i + 1`).
- Bài về **tầng** → BFS với `for _ in range(len(q))`; bài chỉ cần "đầu tiên gặp ở độ sâu d" → DFS tiền thứ tự trái trước phải với `setdefault`.
- `deque` hỗ trợ `q[0]` và `q[-1]` đều `O(1)`: dùng để lấy hai đầu của tầng.
- Ở ngôn ngữ có số nguyên cố định, **chuẩn hóa** chỉ số mỗi tầng (trừ chỉ số trái nhất) để tránh tràn; ở Python thì không bắt buộc.
- `i << 1 | 1` nghĩa là `(i << 1) | 1 = 2i + 1`; nếu cần chắc chắn có thể viết `2 * i + 1`.
- Gốc đánh số `1` (con `2i`, `2i + 1`) là quy ước heap chuẩn; gốc `0` với cùng công thức cũng cho chiều rộng đúng vì ta chỉ so sánh chỉ số trong cùng tầng. Chỉ cần đừng trộn lẫn hai quy ước trong một lời giải.
- Test đừng quên: một nút, dây chuyền trái/phải, cây đầy đủ, cây "hai dây chuyền".

## 15. Biến thể

### 15.1. Chiều rộng của **từng tầng** (danh sách)

Đổi `ans = max(...)` thành thêm vào danh sách.

In [27]:
def level_widths(root):
    res = []
    q = deque([(root, 1)])
    while q:
        res.append(q[-1][1] - q[0][1] + 1)
        for _ in range(len(q)):
            node, i = q.popleft()
            if node.left:
                q.append((node.left, 2 * i))
            if node.right:
                q.append((node.right, 2 * i + 1))
    return res

print(level_widths(build_tree([1, 3, 2, 5, 3, None, 9])))                 # [1, 2, 4]
print(level_widths(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])))  # [1, 2, 2, 7]

[1, 2, 4]
[1, 2, 4, 7]


### 15.2. Độ rộng "chỉ đếm nút thật" (số nút nhiều nhất trên một tầng)

Đây là bài dễ hơn nhiều (không cần chỉ số): `max(len(q))`. So sánh hai định nghĩa trên cùng cây để thấy khác biệt.

In [28]:
def max_nodes_per_level(root):
    ans = 0
    q = deque([root])
    while q:
        ans = max(ans, len(q))
        for _ in range(len(q)):
            node = q.popleft()
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return ans

t = build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])
print("số nút nhiều nhất trên một tầng:", max_nodes_per_level(t), "| chiều rộng theo đề:", Solution().widthOfBinaryTree(t))

số nút nhiều nhất trên một tầng: 2 | chiều rộng theo đề: 7


### 15.3. Bài liên quan dùng chỉ số heap: kiểm tra cây hoàn chỉnh (LeetCode 958)

Một cây nhị phân là **hoàn chỉnh** (complete) khi các nút chiếm các chỉ số `1..n` không có lỗ hổng. Nghĩa là `chỉ số lớn nhất == số nút`. Hãy tự thử viết ở Bài 5 (mục 17).

### 15.4. Các bài liên quan

- LeetCode 102 Binary Tree Level Order Traversal: khung duyệt theo tầng.
- LeetCode 515 Find Largest Value in Each Tree Row, 637 Average of Levels: thống kê theo tầng.
- LeetCode 958 Check Completeness of a Binary Tree: chỉ số heap.
- LeetCode 1161 Maximum Level Sum of a Binary Tree.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 Bung có `None` | liệt kê cả ô trống của từng tầng | mũ theo `n` | mũ theo `n` | chỉ để hiểu đề / oracle cây nhỏ |
| 2 BFS + chỉ số | `(nút, i)`, con `2i`, `2i+1` | `O(n)` | `O(n)` | **cách nộp**, đơn giản, không đệ quy |
| 3 DFS + `first[depth]` | nhớ chỉ số trái nhất mỗi độ sâu | `O(n)` | `O(h)` | tiết kiệm bộ nhớ cho cây rộng; cẩn thận đệ quy sâu |
| 4 BFS chuẩn hóa | trừ chỉ số trái nhất mỗi tầng | `O(n)` | `O(n)` | C++/Java để tránh tràn số |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Gợi ý ở mục 18, đáp án ở mục 19.

### Bài 1 (đoán output)

Dùng `level_widths` ở mục 15.1. Đoán đầu ra cho hai cây sau rồi điền `ex1_answer` (danh sách gồm hai danh sách). **Không** chạy trước khi đoán.

1. `[1, 2, 3, 4, None, None, 5]`
2. `[1, None, 2, None, 3, None, 4]`

In [29]:
ex1_trees = [[1, 2, 3, 4, None, None, 5], [1, None, 2, None, 3, None, 4]]
ex1_answer = None       # ví dụ [[1, 1, 1], [1, 1]]  <- chỉ là mẫu định dạng

### Bài 2 (tính tay)

Tính tay chiều rộng lớn nhất của `[1, 2, 3, 4, None, None, 5, 6, None, None, 7]` bằng cách điền vị trí heap cho từng nút. Điền `ex2_answer` (một số nguyên).

In [30]:
ex2_tree = [1, 2, 3, 4, None, None, 5, 6, None, None, 7]
ex2_answer = None

### Bài 3 (điền TODO)

Hoàn thành `ex3_width`: hai chỗ `TODO` là chỉ số của con trái và con phải. Cell này **không gọi** hàm.

In [31]:
def ex3_width(root):
    ans = 0
    q = deque([(root, 1)])
    while q:
        ans = max(ans, q[-1][1] - q[0][1] + 1)
        for _ in range(len(q)):
            node, i = q.popleft()
            if node.left:
                q.append((node.left, None))     # TODO 1: chỉ số con trái
            if node.right:
                q.append((node.right, None))    # TODO 2: chỉ số con phải
    return ans

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(root)` bằng DFS (Cách 3). Để `pass` nếu chưa làm.

In [32]:
def ex4_rewrite(root):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_is_complete(root)`: trả `True` nếu cây là **cây hoàn chỉnh** (mọi tầng đầy trừ tầng cuối, và tầng cuối dồn sang trái). Gợi ý nhanh: gán chỉ số heap; cây hoàn chỉnh khi chỉ số lớn nhất bằng số nút.

In [33]:
def ex5_is_complete(root):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: tính chỉ số heap của từng nút theo tầng.
- Mức 2: chiều rộng tầng = chỉ số cuối - chỉ số đầu + 1.
- Mức 3: cây 1: tầng 2 có `4` (vị trí 4) và `5` (vị trí 7) nên rộng `4`; cây 2 là dây chuyền phải nên mọi tầng rộng `1`.

**Bài 2**
- Mức 1: vẽ cây đầy đủ và điền vị trí.
- Mức 2: tìm tầng sâu nhất có hai nút xa nhau nhất.
- Mức 3: `6` ở vị trí `8`, `7` ở vị trí `15`, nên rộng `15 - 8 + 1`.

**Bài 3**
- Mức 1: nhớ công thức chỉ số con.
- Mức 2: con trái là `2 * i`.
- Mức 3: `q.append((node.left, 2 * i))` và `q.append((node.right, 2 * i + 1))` (nhớ lấy `i` từ `node, i = q.popleft()`).

**Bài 4**
- Mức 1: `first = {}` và duyệt tiền thứ tự.
- Mức 2: `first.setdefault(depth, i)` rồi cập nhật `best`.
- Mức 3: gọi `dfs(node.left, depth + 1, 2 * i)` rồi `dfs(node.right, depth + 1, 2 * i + 1)`; nhớ nâng `sys.setrecursionlimit` cho cây sâu.

**Bài 5**
- Mức 1: duyệt BFS đánh chỉ số như bài này.
- Mức 2: đếm số nút `n` và ghi chỉ số lớn nhất `m`.
- Mức 3: trả `m == n`. Thử với `[1,2,3,4,5,6]` (đúng) và `[1,2,3,4,5,None,7]` (sai).

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [34]:
ex1_solution = [[1, 2, 4], [1, 1, 1, 1]]
assert [level_widths(build_tree(v)) for v in ex1_trees] == ex1_solution
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: [[1, 2, 4], [1, 1, 1, 1]]


### Đáp án Bài 2

In [35]:
ex2_solution = 8
assert Solution().widthOfBinaryTree(build_tree(ex2_tree)) == ex2_solution
print("Bài 2 OK:", ex2_solution)

Bài 2 OK: 8


### Đáp án Bài 3

In [36]:
def ex3_solution(root):
    ans = 0
    q = deque([(root, 1)])
    while q:
        ans = max(ans, q[-1][1] - q[0][1] + 1)
        for _ in range(len(q)):
            node, i = q.popleft()
            if node.left:
                q.append((node.left, 2 * i))
            if node.right:
                q.append((node.right, 2 * i + 1))
    return ans

rng = random.Random(3)
for _ in range(300):
    t = random_tree(rng.randint(1, 40), rng, max_depth=10, deep_bias=0.4)
    assert ex3_solution(t) == width_path_oracle(t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def ex4_solution(root):
    first = {}
    best = 0
    def dfs(node, depth, i):
        nonlocal best
        if node is None:
            return
        first.setdefault(depth, i)
        best = max(best, i - first[depth] + 1)
        dfs(node.left, depth + 1, 2 * i)
        dfs(node.right, depth + 1, 2 * i + 1)
    dfs(root, 0, 1)
    return best

rng = random.Random(4)
for _ in range(300):
    t = random_tree(rng.randint(1, 60), rng, max_depth=12, deep_bias=0.5)
    assert ex4_solution(t) == width_path_oracle(t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [38]:
def ex5_solution(root):
    n = 0
    biggest = 0
    q = deque([(root, 1)])
    while q:
        node, i = q.popleft()
        n += 1
        biggest = max(biggest, i)
        if node.left:
            q.append((node.left, 2 * i))
        if node.right:
            q.append((node.right, 2 * i + 1))
    return biggest == n


def is_complete_oracle(root):
    # oracle độc lập: duyệt theo tầng với cả None, sau khi gặp None đầu tiên thì không còn nút thật
    q = deque([root])
    seen_none = False
    while q:
        node = q.popleft()
        if node is None:
            seen_none = True
        else:
            if seen_none:
                return False
            q.append(node.left)
            q.append(node.right)
    return True

assert ex5_solution(build_tree([1, 2, 3, 4, 5, 6])) is True
assert ex5_solution(build_tree([1, 2, 3, 4, 5, None, 7])) is False
assert ex5_solution(TreeNode(1)) is True
rng = random.Random(5)
trues = 0
for _ in range(600):
    t = random_tree(rng.randint(1, 15), rng, max_depth=6, deep_bias=rng.choice([0.0, 0.5]))
    assert ex5_solution(t) == is_complete_oracle(t)
    trues += is_complete_oracle(t)
for n in range(1, 30):                                      # cây đầy đủ dạng mảng luôn hoàn chỉnh
    assert ex5_solution(build_tree(list(range(n)))) is True
print("Bài 5 OK (số cây hoàn chỉnh trong ngẫu nhiên:", trues, ")")

Bài 5 OK (số cây hoàn chỉnh trong ngẫu nhiên: 69 )


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from collections import deque

class Solution:
    def widthOfBinaryTree(self, root: Optional[TreeNode]) -> int:
        ans = 0
        q = deque([(root, 1)])                       # (nút, chỉ số heap), gốc = 1
        while q:
            ans = max(ans, q[-1][1] - q[0][1] + 1)   # rộng tầng = cuối - đầu + 1
            for _ in range(len(q)):
                node, i = q.popleft()
                if node.left:
                    q.append((node.left, i << 1))        # 2i
                if node.right:
                    q.append((node.right, i << 1 | 1))   # 2i + 1
        return ans
```

Câu thần chú: **"gốc là 1, con là 2i và 2i+1, rộng = cuối trừ đầu cộng 1, mỗi tầng một lần."**

### Checklist trước khi submit

- [ ] Gốc có chỉ số `1`, con `2i` và `2i + 1`; chỉ so sánh chỉ số **trong cùng một tầng**.
- [ ] Chiều rộng là `cuối - đầu + 1` (có `+ 1`).
- [ ] Đo chiều rộng **trước** khi `popleft` các nút của tầng (lúc `q` chỉ chứa một tầng).
- [ ] Không đẩy nút `None` vào hàng đợi.
- [ ] Nếu dùng DFS: `setdefault` (không ghi đè) và nâng giới hạn đệ quy cho cây sâu `3000`.
- [ ] Ở C++/Java: chuẩn hóa chỉ số mỗi tầng để không tràn.
- [ ] Test: một nút, dây chuyền, hai dây chuyền, cây đầy đủ, ví dụ 1-3 của đề.
- [ ] Skeleton đúng: `class Solution`, method `widthOfBinaryTree(self, root: Optional[TreeNode]) -> int`.